# GitHub Remote MCP + LangGraph — PR Reviewer (No Docker)

This notebook solves the same PR-review problem as the earlier version, but learners do **not** need Docker.

```text
User
  ↓
LangGraph Agent
  ↓
Remote GitHub MCP Server
  ↓
GitHub
```

We use GitHub's hosted MCP endpoint in **read-only mode**.


## 0. Prerequisites

You need only:

1. Python 3.10+
2. OpenAI API key
3. GitHub Personal Access Token with read access to the target repository

Create a `.env` file beside this notebook:

```env
OPENAI_API_KEY=sk-...
GITHUB_PERSONAL_ACCESS_TOKEN=github_pat_...
```

Do **not** commit `.env`.


In [ ]:
# Install dependencies
%pip install -U langchain langchain-openai langgraph langchain-mcp-adapters python-dotenv


In [1]:
# ============================================================
# 1. IMPORTS + ENVIRONMENT
# ============================================================

import os
from typing import TypedDict, Annotated

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_mcp_adapters.client import MultiServerMCPClient

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GITHUB_TOKEN = (
    os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")
    or os.getenv("GITHUB_TOKEN")
)

assert OPENAI_API_KEY, "OPENAI_API_KEY is missing from .env"
assert GITHUB_TOKEN, (
    "Set GITHUB_PERSONAL_ACCESS_TOKEN (recommended) "
    "or GITHUB_TOKEN in .env"
)

print("✅ Environment variables loaded")


/Users/rahultiwari/Documents/02_Freelancing/coding_ninja_fresh/dummy-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Environment variables loaded


In [2]:
# ============================================================
# 2. DEMO CONFIGURATION
# ============================================================

OWNER = "rahul8879"
REPO = "e-comm-agentic-demo"
PR_NUMBER = 10

print(f"Repository : {OWNER}/{REPO}")
print(f"PR number  : #{PR_NUMBER}")


Repository : rahul8879/e-comm-agentic-demo
PR number  : #10


## 3. Connect to GitHub's Remote MCP Server

No local MCP server. No Docker.

The notebook connects to GitHub's hosted MCP endpoint over HTTP using a Bearer token.

We also request only the relevant toolsets and enable read-only mode.


In [13]:
# ============================================================
# 3. REMOTE GITHUB MCP CLIENT
# ============================================================

github_mcp = MultiServerMCPClient(
    {
        "github": {
            "transport": "http",
            "url": "https://api.githubcopilot.com/mcp/",
            "headers": {
                "Authorization": f"Bearer {GITHUB_TOKEN}",
                "X-MCP-Toolsets": "pull_requests,repos,actions",
                "X-MCP-Readonly": "true",
            },
        }
    }
)

tools = await github_mcp.get_tools()

print(f"✅ Loaded {len(tools)} GitHub MCP tools\n")

for tool in tools:
    print(f"• {tool.name}")


✅ Loaded 19 GitHub MCP tools

• actions_get
• actions_list
• get_commit
• get_file_contents
• get_job_logs
• get_latest_release
• get_release_by_tag
• get_tag
• list_branches
• list_commits
• list_pull_requests
• list_releases
• list_repository_collaborators
• list_tags
• pull_request_read
• search_code
• search_commits
• search_pull_requests
• search_repositories


In [4]:
# Optional: inspect PR-related tool descriptions

for tool in tools:
    if "pull" in tool.name.lower() or "pr" in tool.name.lower():
        print("=" * 90)
        print("TOOL:", tool.name)
        print(tool.description[:1500])
        print()


TOOL: list_pull_requests
List pull requests in a GitHub repository. If the user specifies an author, then DO NOT use this tool and use the search_pull_requests tool instead.

TOOL: pull_request_read
Get information on a specific pull request in GitHub repository.

TOOL: search_pull_requests
Search for pull requests in GitHub repositories using issues search syntax already scoped to is:pr



In [5]:
# ============================================================
# 4. LANGGRAPH STATE
# ============================================================

class PRReviewState(TypedDict):
    messages: Annotated[list, add_messages]
    owner: str
    repo: str
    pr_number: int
    verdict: str


In [6]:
# ============================================================
# 5. MODEL + MCP TOOLS
# ============================================================

llm = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
)

llm_with_tools = llm.bind_tools(tools)

print("✅ LLM bound to remote GitHub MCP tools")


✅ LLM bound to remote GitHub MCP tools


In [7]:
# ============================================================
# 6. SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are CodeSentinel, an expert GitHub Pull Request reviewer.

You have access to GitHub tools through MCP.

Your task is to investigate the requested pull request and produce
an evidence-based review.

When useful, inspect:
- PR title and description
- changed files
- diff
- commits
- reviews / review comments
- CI / checks

Rules:
1. Use GitHub MCP tools instead of guessing.
2. Never invent repository, PR, CI, file, review, or code information.
3. This is a read-only review. Never attempt to modify GitHub.
4. Focus on correctness, bugs, security, maintainability, tests,
   backward compatibility, and operational risk.
5. If information is unavailable, say so.

Return the final answer in this structure:

## 🤖 CodeSentinel Review

### 📋 PR Summary
...

### ✅ What Looks Good
- ...

### ⚠️ Issues Found
- ...

### 🔧 Suggestions
- ...

### 🧪 CI / Status
- ...

### 📊 Verdict
APPROVED

or

NEEDS CHANGES

Then give a short reason.
"""


In [8]:
# ============================================================
# 7. AGENT NODE + ROUTING
# ============================================================

async def agent_node(state: PRReviewState):
    task_context = f"""
Target repository: {state['owner']}/{state['repo']}
Target pull request: #{state['pr_number']}
"""

    messages = [
        SystemMessage(content=SYSTEM_PROMPT + "\n" + task_context),
        *state["messages"],
    ]

    response = await llm_with_tools.ainvoke(messages)

    return {"messages": [response]}


def should_continue(state: PRReviewState):
    last_message = state["messages"][-1]

    if getattr(last_message, "tool_calls", None):
        return "tools"

    return END


In [9]:
# ============================================================
# 8. BUILD LANGGRAPH
# ============================================================

tool_node = ToolNode(tools)

builder = StateGraph(PRReviewState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")

builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        END: END,
    },
)

builder.add_edge("tools", "agent")

graph = builder.compile()

print("✅ LangGraph compiled")


✅ LangGraph compiled


In [10]:
# Optional: show Mermaid graph source
print(graph.get_graph().draw_mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 9. Run a real PR review

The LLM decides which MCP tools to call. We do not hard-code a fixed tool sequence.


In [11]:
# ============================================================
# 9. RUN THE REVIEW
# ============================================================

initial_state: PRReviewState = {
    "messages": [
        HumanMessage(
            content=(
                f"Review PR #{PR_NUMBER} in {OWNER}/{REPO}. "
                "Use the available GitHub MCP tools to investigate the PR "
                "and return the final CodeSentinel review."
            )
        )
    ],
    "owner": OWNER,
    "repo": REPO,
    "pr_number": PR_NUMBER,
    "verdict": "",
}

result = await graph.ainvoke(
    initial_state,
    config={"recursion_limit": 30},
)

print("✅ Review complete")


✅ Review complete


In [12]:
# ============================================================
# 10. FINAL REVIEW
# ============================================================

final_review = result["messages"][-1].content
print(final_review)


## 🤖 CodeSentinel Review

### 📋 PR Summary

- **Title:** added the dummy code
- **Description:** Please approve my code
- **Author:** rahul8879
- **Target branch:** main
- **Source branch:** feature/demo
- **Files changed:** 1 (`orders.py`)
- **Commits:** 2
- **Additions/Deletions:** +2/-1
- **CI/Checks:** No check runs found
- **Reviews:** 2 reviews, both requesting changes due to critical security issues

### ✅ What Looks Good

- The PR is small and easy to review.
- The changes are isolated to a single file (`orders.py`).

### ⚠️ Issues Found

**Critical Security Issues (P0/P1):**
- **SQL Injection Vulnerability:** The code constructs SQL queries using string interpolation (f-string) with unsanitized input, making it vulnerable to SQL injection.
- **Hardcoded Secret:** A password is hardcoded in the source code, which is a severe security risk.
- **No Authentication Check:** The endpoint allows any user to access any order, violating basic authentication/authorization principles.
- 

In [ ]:
# ============================================================
# 11. OPTIONAL — SHOW FULL AGENT / TOOL TRACE
# ============================================================

for i, message in enumerate(result["messages"], start=1):
    print("\n" + "=" * 100)
    print(f"MESSAGE {i}: {type(message).__name__}")
    print("=" * 100)

    if getattr(message, "tool_calls", None):
        print("TOOL CALLS:")
        for call in message.tool_calls:
            print(call)

    content = getattr(message, "content", None)
    if content:
        print("\nCONTENT:")
        print(content)
